In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support

data = pd.read_csv("NER dataset.csv", encoding="latin1")
data = data.rename(columns={"Sentence #":"sentence_id","Word":"word","POS":"tag"})
data["sentence_id"] = data["sentence_id"].ffill()

sentences=[]
for _,g in data.groupby("sentence_id"):
    sentences.append(list(zip(g["word"].astype(str), g["tag"].astype(str))))

train_data,test_data=train_test_split(sentences,test_size=0.2,random_state=42)

tags = sorted({t for s in train_data for _, t in s})

words = sorted({str(w).lower() for s in train_data for w, _ in s})

tag_to_idx = {t: i for i, t in enumerate(tags)}

word_to_idx = {w: i for i, w in enumerate(words)}

T = len(tags)
W = len(words)
initial=np.zeros(T)
transition=np.zeros((T,T))
emission=np.zeros((T,W))

for s in train_data:
    initial[tag_to_idx[s[0][1]]]+=1
    for i,(w,t) in enumerate(s):
        ti=tag_to_idx[t]
        wl = str(w).lower()
        if wl in word_to_idx:
            emission[ti,word_to_idx[wl]]+=1
        if i>0:
            pti=tag_to_idx[s[i-1][1]]
            transition[pti,ti]+=1

initial+=1;transition+=1;emission+=1
initial/=initial.sum()
transition/=transition.sum(axis=1,keepdims=True)
emission/=emission.sum(axis=1,keepdims=True)

log_init=np.log(initial)
log_trans=np.log(transition)
log_emit=np.log(emission)
unk=np.log(np.ones(T)/W)

def viterbi(words_sent):

    n = len(words_sent)

    dp = np.full((T, n), -np.inf)
    bp = np.zeros((T, n), dtype=int)

    first_word = str(words_sent[0]).lower()

    if first_word in word_to_idx:
        e = log_emit[:, word_to_idx[first_word]]
    else:
        e = unk

    dp[:, 0] = log_init + e

    for t in range(1, n):

        current_word = str(words_sent[t]).lower()

        if current_word in word_to_idx:
            e = log_emit[:, word_to_idx[current_word]]
        else:
            e = unk

        scores = dp[:, t-1][:, None] + log_trans

        bp[:, t] = np.argmax(scores, axis=0)

        dp[:, t] = np.max(scores, axis=0) + e

    best = np.argmax(dp[:, -1])

    sequence = [best]

    for t in range(n-1, 0, -1):
        best = bp[best, t]
        sequence.append(best)

    sequence.reverse()

    return [tags[i] for i in sequence]

y_true=[];y_pred=[]
for s in test_data:
    ws=[w for w,_ in s]
    ts=[t for _,t in s]
    pr=viterbi(ws)
    y_true.extend(ts); y_pred.extend(pr)

print("Accuracy:",accuracy_score(y_true,y_pred))
p,r,f,_=precision_recall_fscore_support(y_true,y_pred,average="weighted")
print("Precision:",p)
print("Recall:",r)
print("F1:",f)
print(classification_report(y_true,y_pred))

tests=[
"The cat is sleeping",
"She loves programming",
"Birds fly in the sky",
"I bought a new laptop",
"Students are writing exams"
]
for sent in tests:
    ws=sent.split()
    print("\nSentence:",sent)
    for w,t in zip(ws,viterbi(ws)):
        print(f"{w:12} -> {t}")


Accuracy: 0.9320068626983748
Precision: 0.9325143433693424
Recall: 0.9320068626983748
F1: 0.9311262130329577
              precision    recall  f1-score   support

           $       0.98      1.00      0.99       208
           ,       0.98      1.00      0.99      6544
           .       0.95      1.00      0.98      9565
           :       1.00      0.32      0.48       167
           ;       0.95      0.47      0.62        43
          CC       1.00      1.00      1.00      4618
          CD       0.97      0.93      0.95      4983
          DT       0.95      1.00      0.97     19735
          EX       1.00      0.81      0.90       137
          IN       0.96      0.99      0.98     24247
          JJ       0.86      0.91      0.88     15524
         JJR       0.87      0.85      0.86       557
         JJS       0.94      0.88      0.91       638
         LRB       1.00      0.90      0.95       137
          MD       0.97      0.99      0.98      1347
          NN       0.91   